# IF3270 TB2 Experiments

Notebook runner for the required CNN and RNN/LSTM experiment artifacts. The training and evaluation logic lives in `scripts/`; this notebook makes runs reproducible and saveable by writing weights, histories, evaluation JSON, plots, and `artifacts/summary.csv`.

## Usage

1. Set the dataset paths in the configuration cell.
2. Run dry-run cells first to inspect commands.
3. Set `RUN_HEAVY = True` before running training/evaluation cells.

Large raw datasets should stay outside the repository. Saved deliverables go under `artifacts/`.

In [ ]:
from __future__ import annotations

import csv
import os
import shlex
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path.cwd().parents[1]
os.chdir(ROOT)

ARTIFACTS = ROOT / "artifacts"
CNN_ARTIFACTS = ARTIFACTS / "cnn"
CAPTION_ARTIFACTS = ARTIFACTS / "captioning"
PLOTS_DIR = ARTIFACTS / "plots"
for path in (ARTIFACTS, CNN_ARTIFACTS, CAPTION_ARTIFACTS, PLOTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Leave these as None to let the scripts download datasets with KaggleHub.
# Set them to local paths only if the datasets are already available there.
INTEL_ROOT = None
FLICKR_ROOT = None
FLICKR_IMAGES = None
FLICKR_CAPTIONS = None
FLICKR_TRAIN_SPLIT = None
FLICKR_TEST_SPLIT = None

FEATURES = ARTIFACTS / "flickr8k_features.npy"
FEATURE_PATHS = ARTIFACTS / "flickr8k_paths.txt"
VOCAB = ARTIFACTS / "vocab.json"
CAPTIONS_JSON = ARTIFACTS / "captions_tokens.json"

RUN_HEAVY = True
FAST_MODE = True
CNN_TARGET_SIZE = (64, 64)
CNN_DENSE_UNITS = 32
CAPTION_HIDDEN_SIZES = (32, 64)
CAPTION_EMBED_DIM = 64
CAPTION_MAX_LENGTH = 12
FEATURE_LIMIT = None
FEATURE_ENCODER = "mobilenet_v2"


def command(*parts: object) -> list[str]:
    return [str(part) for part in parts if part is not None]


def with_optional_path(cmd: list[str], flag: str, path: Path | str | None) -> list[str]:
    if path is None:
        return cmd
    return cmd + [flag, str(path)]


def show(cmd: list[str]) -> None:
    print(" ".join(shlex.quote(part) for part in cmd))


def run(cmd: list[str], *, heavy: bool = False) -> subprocess.CompletedProcess[str] | None:
    show(cmd)
    # if heavy and not RUN_HEAVY:
    #     print("Skipped. Set RUN_HEAVY = True to execute this cell.")
    #     return None
    return subprocess.run(cmd, cwd=ROOT, check=True, text=True)

ROOT


## Experiment Matrix

This expands the required CNN and captioning variations from the task specification.

In [ ]:
matrix_cmd = command(
    "uv", "run", "python", "scripts/run_experiment_matrix.py",
    "--suite", "all",
    "--dry-run",
    "--features", FEATURES,
    "--paths-file", FEATURE_PATHS,
    "--vocab", VOCAB,
    "--captions-json", CAPTIONS_JSON,
    "--cnn-output", CNN_ARTIFACTS,
    "--caption-output", CAPTION_ARTIFACTS,
)
matrix_cmd = with_optional_path(matrix_cmd, "--intel-root", INTEL_ROOT)
run(matrix_cmd, heavy=True)


## CNN Training

Each run saves `*.weights.h5` and `*.history.json` under `artifacts/cnn/`.

In [ ]:
cnn_example = command(
    "uv", "run", "python", "scripts/train_cnn.py",
    "--output-dir", CNN_ARTIFACTS,
    "--layer-type", "conv2d",
    "--num-conv-layers", "1",
    "--filters", "8",
    "--kernel-size", "3",
    "--pooling", "max",
    "--epochs", "5",
    "--batch-size", "8",
    "--target-size", CNN_TARGET_SIZE[0], CNN_TARGET_SIZE[1],
    "--dense-units", CNN_DENSE_UNITS,
)
cnn_example = with_optional_path(cnn_example, "--data-root", INTEL_ROOT)
run(cnn_example, heavy=True)


Run the full CNN matrix after checking the dry run. This cell saves all training histories and weights.

In [ ]:
cnn_matrix_cmd = command(
    "uv", "run", "python", "scripts/run_experiment_matrix.py",
    "--suite", "cnn",
    "--cnn-output", CNN_ARTIFACTS,
    "--cnn-epochs", "5",
    "--batch-size", "8",
    "--cnn-target-size", CNN_TARGET_SIZE[0], CNN_TARGET_SIZE[1],
    "--cnn-dense-units", CNN_DENSE_UNITS,
    "--caption-hidden-sizes", CAPTION_HIDDEN_SIZES[0], CAPTION_HIDDEN_SIZES[1],
    "--caption-embed-dim", CAPTION_EMBED_DIM,
    "--caption-max-length", CAPTION_MAX_LENGTH,
)
cnn_matrix_cmd = with_optional_path(cnn_matrix_cmd, "--intel-root", INTEL_ROOT)
run(cnn_matrix_cmd, heavy=True)


## CNN Scratch vs Keras Evaluation

Evaluate the best shared Conv2D training artifact, then evaluate the matching non-shared LocallyConnected2D architecture. The output JSON files are picked up by the summary script.

In [ ]:
best_cnn_tags = [
    "conv2d_3x8-12-16_k3_max",
    "locally_connected_3x8-12-16_k3_max",
]

for best_cnn_tag in best_cnn_tags:
    cnn_eval_cmd = command(
        "uv", "run", "python", "scripts/eval_cnn_scratch.py",
        "--weights", CNN_ARTIFACTS / f"{best_cnn_tag}.weights.h5",
        "--config", CNN_ARTIFACTS / f"{best_cnn_tag}.history.json",
        "--output", CNN_ARTIFACTS / f"eval_{best_cnn_tag}.json",
        "--batch-size", "8",
    )
    cnn_eval_cmd = with_optional_path(cnn_eval_cmd, "--data-root", INTEL_ROOT)
    run(cnn_eval_cmd, heavy=True)


## Flickr8k Preprocessing

Feature extraction and vocabulary building are one-time steps. They save reusable `.npy`, `.txt`, and `.json` artifacts.

In [17]:
feature_cmd = command(
    "uv", "run", "python", "scripts/extract_features.py",
    "--output", FEATURES,
    "--paths-output", FEATURE_PATHS,
    "--encoder", FEATURE_ENCODER,
    "--batch-size", "32",
)
feature_cmd = with_optional_path(feature_cmd, "--limit", FEATURE_LIMIT)
feature_cmd = with_optional_path(feature_cmd, "--flickr-root", FLICKR_ROOT)
feature_cmd = with_optional_path(feature_cmd, "--images-dir", FLICKR_IMAGES)
run(feature_cmd, heavy=True)

vocab_cmd = command(
    "uv", "run", "python", "scripts/build_vocab.py",
    "--output", VOCAB,
    "--captions-output", CAPTIONS_JSON,
    "--image-list", FEATURE_PATHS,
    "--min-count", "2",
)
vocab_cmd = with_optional_path(vocab_cmd, "--flickr-root", FLICKR_ROOT)
vocab_cmd = with_optional_path(vocab_cmd, "--captions", FLICKR_CAPTIONS)
vocab_cmd = with_optional_path(vocab_cmd, "--train-split", FLICKR_TRAIN_SPLIT)
run(vocab_cmd, heavy=True)


uv run python scripts/extract_features.py --output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy --paths-output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt --encoder mobilenet_v2 --batch-size 32
Path to dataset files: /Users/ellipsis/.cache/kagglehub/datasets/adityajn105/flickr8k/versions/1


/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/scripts/extract_features.py:26: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = tf.keras.applications.MobileNetV2(include_top=False, pooling="avg", weights="imagenet")


Saved 8091 features of dim 1280 to /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy
uv run python scripts/build_vocab.py --output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json --captions-output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json --image-list /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt --min-count 2
Path to dataset files: /Users/ellipsis/.cache/kagglehub/datasets/adityajn105/flickr8k/versions/1
Vocabulary size: 5156
Images: 8091  Captions: 40455


CompletedProcess(args=['uv', 'run', 'python', 'scripts/build_vocab.py', '--output', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json', '--captions-output', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json', '--image-list', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt', '--min-count', '2'], returncode=0)

## Captioning Training

The full matrix covers RNN and LSTM, recurrent layer counts 1/2/3, and hidden sizes 32/64. Each run saves weights and history under `artifacts/captioning/`.

In [18]:
run(command(
    "uv", "run", "python", "scripts/run_experiment_matrix.py",
    "--suite", "caption",
    "--features", FEATURES,
    "--paths-file", FEATURE_PATHS,
    "--vocab", VOCAB,
    "--captions-json", CAPTIONS_JSON,
    "--caption-output", CAPTION_ARTIFACTS,
    "--caption-epochs", "5",
    "--batch-size", "8",
    "--caption-hidden-sizes", CAPTION_HIDDEN_SIZES[0], CAPTION_HIDDEN_SIZES[1],
    "--caption-embed-dim", CAPTION_EMBED_DIM,
    "--caption-max-length", CAPTION_MAX_LENGTH,
), heavy=True)


uv run python scripts/run_experiment_matrix.py --suite caption --features /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy --paths-file /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt --vocab /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json --captions-json /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json --caption-output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning --caption-epochs 5 --batch-size 8 --caption-hidden-sizes 32 64 --caption-embed-dim 64 --caption-max-length 12
uv run python scripts/train_caption.py --features /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy --paths-file /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt --vocab /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json --captions-json /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json --output-dir /

CompletedProcess(args=['uv', 'run', 'python', 'scripts/run_experiment_matrix.py', '--suite', 'caption', '--features', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy', '--paths-file', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt', '--vocab', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json', '--captions-json', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json', '--caption-output', '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning', '--caption-epochs', '5', '--batch-size', '8', '--caption-hidden-sizes', '32', '64', '--caption-embed-dim', '64', '--caption-max-length', '12'], returncode=0)

## Captioning Evaluation

Evaluate the best RNN and LSTM artifacts. Use `--mode both` for Keras vs scratch comparison and greedy decoding. Use `--mode scratch --decoding beam` for beam search bonus experiments.

In [19]:
caption_eval_jobs = [
    ("rnn_L1_H32", "greedy", "both", 3, 20),
    ("lstm_L1_H32", "greedy", "both", 3, 20),
    ("lstm_L1_H32", "beam", "scratch", 3, 20),
]

test_split_arg = FLICKR_TEST_SPLIT
for tag, decoding, mode, beam_width, max_length in caption_eval_jobs:
    eval_cmd = command(
        "uv", "run", "python", "scripts/eval_caption.py",
        "--weights", CAPTION_ARTIFACTS / f"{tag}.weights.h5",
        "--config", CAPTION_ARTIFACTS / f"{tag}.history.json",
        "--vocab", VOCAB,
        "--captions-json", CAPTIONS_JSON,
        "--features", FEATURES,
        "--paths-file", FEATURE_PATHS,
        "--mode", mode,
        "--decoding", decoding,
        "--beam-width", beam_width,
        "--max-length", max_length,
        "--output", CAPTION_ARTIFACTS / f"eval_{tag}_{mode}_{decoding}_max{max_length}.json",
        "--num-examples", "10",
    )
    eval_cmd = with_optional_path(eval_cmd, "--test-split", test_split_arg)
    run(eval_cmd, heavy=True)


uv run python scripts/eval_caption.py --weights /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/rnn_L1_H32.weights.h5 --config /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/rnn_L1_H32.history.json --vocab /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json --captions-json /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json --features /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy --paths-file /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt --mode both --decoding greedy --beam-width 3 --max-length 20 --output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/eval_rnn_L1_H32_both_greedy_max20.json --num-examples 10
scratch BLEU-4:  0.0000
scratch METEOR:  0.1389
scratch seconds/image: 0.0012
keras BLEU-4:  0.0000
keras METEOR:  0.1389
keras seconds/image: 0.1896
---
image: 1000268201_693b08cb0e.jpg
scratch: man a in and are in of
keras

## Maximum Caption Length Variation

After choosing the best architecture, run at least three maximum-length variants and compare BLEU-4.

In [20]:
best_caption_tag = "lstm_L1_H32"
for max_length in (12, 20, 28):
    eval_cmd = command(
        "uv", "run", "python", "scripts/eval_caption.py",
        "--weights", CAPTION_ARTIFACTS / f"{best_caption_tag}.weights.h5",
        "--config", CAPTION_ARTIFACTS / f"{best_caption_tag}.history.json",
        "--vocab", VOCAB,
        "--captions-json", CAPTIONS_JSON,
        "--features", FEATURES,
        "--paths-file", FEATURE_PATHS,
        "--mode", "scratch",
        "--decoding", "greedy",
        "--max-length", max_length,
        "--output", CAPTION_ARTIFACTS / f"eval_{best_caption_tag}_scratch_greedy_max{max_length}.json",
    )
    eval_cmd = with_optional_path(eval_cmd, "--test-split", FLICKR_TEST_SPLIT)
    run(eval_cmd, heavy=True)


uv run python scripts/eval_caption.py --weights /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/lstm_L1_H32.weights.h5 --config /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/lstm_L1_H32.history.json --vocab /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/vocab.json --captions-json /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captions_tokens.json --features /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_features.npy --paths-file /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/flickr8k_paths.txt --mode scratch --decoding greedy --max-length 12 --output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/eval_lstm_L1_H32_scratch_greedy_max12.json
scratch BLEU-4:  0.0000
scratch METEOR:  0.1654
scratch seconds/image: 0.0015
---
image: 1000268201_693b08cb0e.jpg
scratch: man a and are on in of
ref:   a child in a pink dress is climbing up a set of stairs in an entry way
ref:   a girl going int

## Summary Tables and Plots

This cell converts saved JSON artifacts into a CSV table and loss plots for the report.

In [22]:
run(command(
    "uv", "run", "python", "scripts/summarize_results.py",
    "--artifacts-dir", ARTIFACTS,
    "--output", ARTIFACTS / "summary.csv",
    "--plot-losses",
    "--plots-dir", PLOTS_DIR,
))

summary_path = ARTIFACTS / "summary.csv"
if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    print(f"Rows: {len(rows)}")
    for row in rows[:10]:
        print(row)

uv run python scripts/summarize_results.py --artifacts-dir /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts --output /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/summary.csv --plot-losses --plots-dir /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/plots
Wrote 30 rows to /Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/summary.csv
Rows: 30
{'file': '/Users/ellipsis/Documents/Seq2SeqIsAllYouNeed/artifacts/captioning/eval_lstm_L1_H32_both_greedy_max20.json', 'kind': 'caption_eval', 'layer_type': '', 'rnn_type': '', 'num_conv_layers': '', 'num_layers': '', 'filters': '', 'kernel_size': '', 'pooling': '', 'hidden_size': '', 'max_length': '20', 'mode': 'scratch', 'decoding': 'greedy', 'test_macro_f1': '', 'keras_macro_f1': '', 'scratch_macro_f1': '', 'prediction_agreement': '', 'bleu_4': '1.1666701713586031e-05', 'meteor': '0.16542126649861166', 'seconds_per_image': '0.001949512564744699', 'num_evaluated': '8091', 'final_loss': '', 'final_val_loss': ''}
{'fil